# Hybrid Skill Gap Analysis

This notebook builds a reusable matcher from the actual `skills`, `Job Title`, and `Role` fields in the job-posting dataset. It applies exact matching first, then RapidFuzz, then optional Sentence Transformer similarity. The transformer is not serialized into the joblib artifact.

## Load and inspect the job corpus

The 50,000-row CSV is read with pandas' CSV parser. Contact fields are not used.

In [ ]:
from pathlib import Path
import re
import unicodedata
import joblib
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

PROJECT_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'ml' / 'data').exists())
DATA_PATH = PROJECT_ROOT / 'ml' / 'data' / 'job_descriptions.csv'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), DATA_PATH
df = pd.read_csv(DATA_PATH, low_memory=False)
print('shape:', df.shape)
print('columns:', df.columns.tolist())
display(df[['Job Title', 'Role', 'skills']].head())
print('missing values:', df.isna().sum().loc[lambda values: values.gt(0)].to_dict())
print('duplicate rows:', int(df.duplicated().sum()))
print('unique job titles:', int(df['Job Title'].nunique()))

## Normalize and extract observed skills

The source uses inconsistent separators and occasionally contains encoding artifacts. A conservative vocabulary is built from terms that actually occur in the `skills` corpus. Safe aliases are applied only when their canonical term is observed.

In [ ]:
def normalize_text(value):
    text = '' if pd.isna(value) else str(value)
    text = text.replace('â€™', "'").replace('â€“', '-')
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_skill(value):
    return re.sub(r'[^a-z0-9+#.]+', ' ', normalize_text(value)).strip()

safe_aliases = {
    'ml': 'machine learning', 'machine-learning': 'machine learning',
    'ai': 'artificial intelligence', 'powerbi': 'power bi',
    'ms excel': 'excel', 'microsoft excel': 'excel',
    'js': 'javascript', 'postgresql': 'postgres',
}
candidate_terms = [
    'machine learning', 'artificial intelligence', 'deep learning', 'data analysis', 'data visualization',
    'project management', 'problem solving', 'communication skills', 'time management', 'critical thinking',
    'social media', 'content creation', 'customer service', 'financial analysis', 'business analysis',
    'statistical analysis', 'database management', 'data management', 'digital marketing', 'market research',
    'python', 'sql', 'excel', 'tableau', 'power bi', 'r programming', 'java', 'javascript', 'typescript',
    'html', 'css', 'react', 'angular', 'node.js', 'c++', 'c#', 'php', 'ruby', 'go', 'matlab', 'sas',
    'tensorflow', 'pytorch', 'scikit learn', 'spark', 'hadoop', 'aws', 'azure', 'docker', 'git', 'linux',
    'autocad', 'accounting', 'sales', 'recruitment', 'seo', 'advertising', 'copywriting',
]
corpus_text = ' '.join(df['skills'].map(normalize_text))
observed_terms = []
for term in sorted(candidate_terms, key=len, reverse=True):
    if re.search(r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])', corpus_text):
        observed_terms.append(term)
observed_terms = sorted(set(observed_terms), key=lambda value: (len(value), value))
aliases = {key: value for key, value in safe_aliases.items() if value in observed_terms}
print('observed canonical terms:', len(observed_terms))
print(observed_terms)

In [ ]:
def extract_skills(value):
    text = normalize_skill(value)
    found = []
    for term in sorted(observed_terms, key=len, reverse=True):
        if re.search(r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])', text):
            found.append(term)
    if found:
        return sorted(set(found))
    fragments = re.split(r'[,;|/]+', text)
    return sorted({fragment.strip() for fragment in fragments if len(fragment.strip()) > 1})

def normalize_skill_list(values):
    if isinstance(values, str):
        values = re.split(r'[,;|/]+', values)
    normalized = []
    for value in values or []:
        item = normalize_skill(value)
        item = aliases.get(item, item)
        if item:
            normalized.append(item)
    return sorted(set(normalized))

df['normalized_title'] = df['Job Title'].map(normalize_text)
df['normalized_role'] = df['Role'].map(normalize_text)
df['extracted_skills'] = df['skills'].map(extract_skills)
requirements_by_title = (df.groupby('normalized_title')['extracted_skills']
    .apply(lambda series: sorted(set(skill for skills in series for skill in skills)))
    .to_dict())
title_to_display = df.drop_duplicates('normalized_title').set_index('normalized_title')['Job Title'].to_dict()
print('titles with requirements:', len(requirements_by_title))
display(df[['Job Title', 'skills', 'extracted_skills']].head(3))

## Matching pipeline and transparent evaluation

Exact matches are authoritative. Fuzzy matches are accepted only above a strong threshold. Semantic matching is attempted only for remaining skills and only when the optional pretrained model is available. The evaluation uses requirements extracted from held-out job rows as transparent proxy labels, not as a claim of real-world benchmark performance.

In [ ]:
FUZZY_THRESHOLD = 90
SEMANTIC_THRESHOLD = 0.72
MODEL_NAME = 'all-MiniLM-L6-v2'
semantic_model = None
semantic_status = 'unavailable'
try:
    from sentence_transformers import SentenceTransformer
    semantic_model = SentenceTransformer(MODEL_NAME)
    semantic_status = 'loaded'
except Exception as error:
    print('Semantic model unavailable; using exact/fuzzy fallback:', type(error).__name__)
print('semantic status:', semantic_status)

In [ ]:
def hybrid_match(candidate_skills, required_skills, fuzzy_threshold=FUZZY_THRESHOLD, semantic_threshold=SEMANTIC_THRESHOLD):
    candidates = normalize_skill_list(candidate_skills)
    required = normalize_skill_list(required_skills)
    matched = set()
    methods = {}
    for requirement in required:
        if requirement in candidates:
            matched.add(requirement)
            methods[requirement] = 'exact'
    remaining_candidates = [item for item in candidates if item not in matched]
    for requirement in required:
        if requirement in matched:
            continue
        if not remaining_candidates:
            break
        best = process.extractOne(requirement, remaining_candidates, scorer=fuzz.token_set_ratio)
        if best and best[1] >= fuzzy_threshold:
            matched.add(requirement)
            methods[requirement] = 'fuzzy'
            remaining_candidates.remove(best[0])
    if semantic_model is not None:
        unmatched_requirements = [item for item in required if item not in matched]
        if unmatched_requirements and remaining_candidates:
            requirement_vectors = semantic_model.encode(unmatched_requirements, normalize_embeddings=True)
            candidate_vectors = semantic_model.encode(remaining_candidates, normalize_embeddings=True)
            scores = requirement_vectors @ candidate_vectors.T
            for row_index, requirement in enumerate(unmatched_requirements):
                best_index = int(np.argmax(scores[row_index]))
                if float(scores[row_index, best_index]) >= semantic_threshold:
                    matched.add(requirement)
                    methods[requirement] = 'semantic'
    matched_list = sorted(matched)
    missing_list = sorted(set(required) - matched)
    return matched_list, missing_list, methods

def analyze_skill_gap(candidate_skills, target_job):
    title = normalize_text(target_job)
    required = requirements_by_title.get(title)
    if required is None:
        return {'target_job': target_job, 'job_found': False, 'required_skills': [], 'matched_skills': [], 'missing_skills': [], 'skill_match_percentage': 0.0, 'recommended_skills': []}
    matched, missing, methods = hybrid_match(candidate_skills, required)
    percentage = round(100 * len(matched) / len(required), 2) if required else 0.0
    return {'target_job': title_to_display.get(title, target_job), 'job_found': True, 'required_skills': required, 'matched_skills': matched, 'missing_skills': missing, 'skill_match_percentage': percentage, 'recommended_skills': missing, 'match_methods': methods}

known_title = next(title for title, skills in requirements_by_title.items() if skills)
known_requirements = requirements_by_title[known_title]
known_result = analyze_skill_gap(known_requirements[:2], known_title)
unknown_result = analyze_skill_gap(['python'], 'Job Title That Does Not Exist')
print(known_result)
print(unknown_result)

In [ ]:
# Proxy evaluation: held-out rows provide requirements and candidate skills from the same posting.
evaluation = []
for row in df.sample(min(100, len(df)), random_state=42).itertuples():
    truth = set(row.extracted_skills)
    if not truth:
        continue
    predicted, _, _ = hybrid_match(row.extracted_skills, truth)
    predicted = set(predicted)
    evaluation.append({'precision': len(predicted & truth) / len(predicted) if predicted else 0.0, 'recall': len(predicted & truth) / len(truth), 'f1': 2 * len(predicted & truth) / (len(predicted) + len(truth)) if predicted else 0.0})
evaluation_df = pd.DataFrame(evaluation)
print('proxy evaluation rows:', len(evaluation_df))
display(evaluation_df.mean().to_frame('mean'))
print('Note: this proxy checks matcher consistency, not independent real-world accuracy.')

## Save configuration artifact

The artifact stores normalized requirements, safe aliases, thresholds, and the transformer model name. The transformer weights remain external and can be loaded by a future engine when available.

In [ ]:
artifact = {
    'artifact_type': 'hybrid_skill_gap_configuration',
    'version': 1,
    'source_file': DATA_PATH.name,
    'requirements_by_title': requirements_by_title,
    'title_to_display': title_to_display,
    'observed_terms': observed_terms,
    'aliases': aliases,
    'fuzzy_threshold': FUZZY_THRESHOLD,
    'semantic_threshold': SEMANTIC_THRESHOLD,
    'semantic_model_name': MODEL_NAME,
    'semantic_status_at_build': semantic_status,
    'matching_order': ['exact', 'fuzzy', 'semantic'],
    'sensitive_columns_excluded': ['Contact Person', 'Contact'],
}
artifact_path = MODEL_DIR / 'skill_gap_model.joblib'
joblib.dump(artifact, artifact_path)
assert artifact_path.exists() and artifact_path.stat().st_size > 0
loaded_artifact = joblib.load(artifact_path)
assert loaded_artifact['artifact_type'] == 'hybrid_skill_gap_configuration'
assert loaded_artifact['requirements_by_title']
print('saved:', artifact_path)
print('titles saved:', len(loaded_artifact['requirements_by_title']))